# Token cost of human-romanized Hindi

Run the cells from top to bottom on Kaggle or Colab (CPU is enough).

Before you start:
- **Kaggle:** switch on Internet in the notebook settings.
- **Gated tokenizers (Llama 2, Llama 3, Llama 4, Gemma 3):** accept each license on Hugging Face, then save a
  free read token as a notebook secret named `HF_TOKEN`. Without it those four are skipped and logged.

Everything is written to `results/` and `data/`. The last cell packs them into `outputs.zip`.


## Stage 1: install, download Dakshina, print the Hindi files

In [ ]:
# Stage 1: install dependencies, log versions, download Dakshina, print the Hindi file layout.
import collections
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from importlib.metadata import PackageNotFoundError, version

URL = "https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar"
TAR = "dakshina_dataset_v1.0.tar"
LANG = "hi"
PKGS = ["numpy", "pandas", "scipy", "matplotlib", "transformers", "tokenizers", "huggingface_hub",
        "tiktoken", "sentencepiece", "protobuf", "indic-transliteration"]

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))

# No version pins: pins would fight the packages Kaggle/Colab preinstall.
# The versions that actually ran are logged below and written to requirements.txt.
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], capture_output=True, text=True)
if pip.returncode:
    sys.exit("pip install failed:\n" + (pip.stdout + pip.stderr)[-3000:])


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


info = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {p: installed(p) for p in PKGS},
}

with urllib.request.urlopen(urllib.request.Request(URL, method="HEAD")) as r:
    size = int(r.headers["Content-Length"])
print(f"Dakshina archive: {size / 1e9:.2f} GB | free disk here: {shutil.disk_usage('.').free / 1e9:.1f} GB")
if not os.path.exists(TAR) or os.path.getsize(TAR) != size:
    urllib.request.urlretrieve(URL, TAR)

sha = hashlib.sha256()
with open(TAR, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        sha.update(chunk)
info["dakshina"] = {"url": URL, "bytes": size, "sha256": sha.hexdigest()}

with tarfile.open(TAR) as tar:
    members = tar.getmembers()
    print("\nTop of archive:", sorted({"/".join(m.name.split("/")[:2]) for m in members}))
    hindi = [m for m in members if LANG in m.name.split("/")]
    print(f"\n=== Everything under '{LANG}' (bytes, path) ===")
    for m in hindi:
        print(f"{m.size:>12,}  {m.name}{'/' if m.isdir() else ''}")
    romanized = [m for m in hindi if m.isfile() and "romanized" in m.name.split("/")]
    tar.extractall("data", members=romanized, filter="data")

print("\n=== Romanized files: line count, tab-separated fields per line, first 5 lines (cut at 200 chars) ===")
for m in romanized:
    # newline="" keeps the original line endings visible in repr()
    with open(os.path.join("data", m.name), encoding="utf-8", errors="replace", newline="") as f:
        lines = f.readlines()
    fields = collections.Counter(line.count("\t") + 1 for line in lines)
    print(f"\n--- {m.name}\nlines={len(lines)} fields_per_line={dict(fields)}")
    for line in lines[:5]:
        print(repr(line[:200]))

with open("versions.json", "w") as f:
    json.dump(info, f, indent=2)
with open("requirements.txt", "w") as f:
    f.write("".join(f"{p}=={v}\n" for p, v in info["packages"].items() if v))
print("\n=== versions.json ===\n" + json.dumps(info, indent=2))


## Stage 2: build the four conditions (slow cell: IndicXlit)

In [ ]:
# Stage 2: build the four conditions (DEV, HUM, XLIT, ITR) for the same Hindi sentences.
# Writes data/dakshina_hi_4cond.tsv and results/stage2_report.json.
import glob
import json
import os
import shutil
import subprocess
import sys
import unicodedata

import pandas as pd
from indic_transliteration import sanscript

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))
os.makedirs("results", exist_ok=True)

ROM = "data/dakshina_dataset_v1.0/hi/romanized/hi.romanized.rejoined"
MIN_WORDS = 5
# IndicXlit needs fairseq, which only ships ready-made builds up to Python 3.8,
# so it runs in its own environment and the notebook's packages are left alone.
ENV = os.environ.get("XLIT_ENV", "/tmp/xlit_env")
PY = f"{ENV}/bin/python"


def read_aligned(path):
    """One (native, latin) word pair per line; a '</s>' pair ends each sentence."""
    sentences, current = [], []
    with open(path, encoding="utf-8", newline="\n") as f:
        for n, line in enumerate(f, 1):
            fields = line.rstrip("\n").split("\t")
            if len(fields) != 2:
                sys.exit(f"{path} line {n}: expected 2 tab-separated fields, got {len(fields)}")
            if fields[0] == "</s>":
                sentences.append(current)
                current = []
            else:
                current.append(fields)
    if current:
        sys.exit(f"{path}: last sentence has no closing </s>")
    return sentences


# Main text: Dakshina's own punctuation-free, lowercased, word-aligned file.
# Raw text: the cased, punctuated sentences, kept as a robustness check.
nopunct = read_aligned(ROM + ".aligned.cased_nopunct.tsv")
cased = read_aligned(ROM + ".aligned.tsv")
with open(ROM + ".tsv", encoding="utf-8", newline="\n") as f:
    raw = [line.rstrip("\n").split("\t") for line in f]

# The three files must hold the same sentences in the same order.
assert all(len(r) == 2 for r in raw), "rejoined.tsv: expected 2 fields on every line"
assert len(nopunct) == len(cased) == len(raw), "the three files have different sentence counts"
assert all(len(a) == len(b) for a, b in zip(nopunct, cased)), "aligned files disagree on word counts"
assert all(
    " ".join(w[0] for w in s) == r[0] and " ".join(w[1] for w in s) == r[1] for s, r in zip(cased, raw)
), "aligned.tsv does not join back to rejoined.tsv"


def nfc(text):
    return unicodedata.normalize("NFC", text)


df = pd.DataFrame({
    "id": range(1, len(raw) + 1),
    "dev": [" ".join(w[0] for w in s) for s in nopunct],
    "hum": [" ".join(w[1] for w in s) for s in nopunct],
    "dev_raw": [r[0] for r in raw],
    "hum_raw": [r[1] for r in raw],
})
report = {"source_sentences": len(df)}
for col in ["dev", "hum", "dev_raw", "hum_raw"]:
    fixed = df[col].map(nfc)
    report[f"changed_by_nfc_{col}"] = int((fixed != df[col]).sum())
    df[col] = fixed
lowered = df["hum"].str.lower()
report["changed_by_lowercasing_hum"] = int((lowered != df["hum"]).sum())
df["hum"] = lowered

dev_words = df["dev"].str.split().str.len()
report["dropped_under_%d_words" % MIN_WORDS] = int((dev_words < MIN_WORDS).sum())
df = df[dev_words >= MIN_WORDS].reset_index(drop=True)
dev_words = df["dev"].str.split().str.len()
report["remaining_sentences"] = len(df)
report["remaining_dev_words"] = int(dev_words.sum())
report["counted_but_not_filtered"] = {
    "duplicate_dev_sentences": int(df["dev"].duplicated().sum()),
    "dev_with_latin_letters": int(df["dev"].str.contains("[A-Za-z]").sum()),
    "dev_with_digits": int(df["dev"].str.contains(r"\d").sum()),
    "dev_with_zero_width_joiners": int(df["dev"].str.contains("[‌‍]").sum()),
    "hum_word_count_differs_from_dev": int((df["hum"].str.split().str.len() != dev_words).sum()),
}

df["itr_cased"] = [sanscript.transliterate(s, sanscript.DEVANAGARI, sanscript.ITRANS) for s in df["dev"]]
df["itr"] = df["itr_cased"].str.lower()

# Runs inside the Python 3.8 environment. Transliterates each distinct word once, in batches.
WORKER = r'''
import json, logging, sys, time
from importlib.metadata import version
import fairseq, numpy, torch
from ai4bharat.transliteration import XlitEngine
from ai4bharat.transliteration.transformer import indic2en
from ai4bharat.transliteration.transformer.base_engine import LANG_WORD_REGEXES
from ai4bharat.transliteration.utils import INDIC_TO_LATIN_PUNCT_TRANSLATOR, INDIC_TO_STANDARD_NUMERALS_TRANSLATOR

logging.disable(logging.INFO)  # fairseq logs several lines per batch
src, dst, cache_path = sys.argv[1:4]
BEAM, BATCH, PARITY_N = 4, 256, 20
# column -> (rescore, candidates per word).
# "xlit" is the model's own best guess, the setting in the IndicXlit README example for Indic-to-Roman.
# "xlit_rescored" is the library default: the top 4 guesses re-ranked with a word-frequency dictionary.
VARIANTS = {"xlit": (False, 1), "xlit_rescored": (True, 4)}
WORD = LANG_WORD_REGEXES["hi"]


def prep(s):
    # the same steps IndicXlit applies inside its own sentence function
    return s.lower().strip().translate(INDIC_TO_LATIN_PUNCT_TRANSLATOR).translate(INDIC_TO_STANDARD_NUMERALS_TRANSLATOR)


with open(src, encoding="utf-8") as f:
    sentences = json.load(f)
prepped = [prep(s) for s in sentences]
words = sorted({w for s in prepped for w in WORD.findall(s)})
try:
    with open(cache_path, encoding="utf-8") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}

start = time.time()
columns, variants = {}, {}
for column, (rescore, topk) in VARIANTS.items():
    done = cache.setdefault(column, {})
    todo = [w for w in words if w not in done]
    print("IndicXlit %s: %d distinct words, %d still to do" % (column, len(words), len(todo)), flush=True)
    engine = XlitEngine(src_script_type="indic", beam_width=BEAM, rescore=rescore)
    began = time.time()
    for i in range(0, len(todo), BATCH):
        chunk = todo[i:i + BATCH]
        best = engine.batch_transliterate_words(chunk, "hi", "en", topk=topk)[0]
        if len(best) == topk * len(chunk):
            best = best[::topk]
        else:
            # batch output is not `topk` per word: use the public one-word call instead
            best = [engine.translit_word(w, "hi", topk=topk)[0] for w in chunk]
        done.update(zip(chunk, best))
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
        n = i + len(chunk)
        if n == len(todo) or (i // BATCH) % 10 == 9:
            print("  %d/%d words, %.0f words/sec" % (n, len(todo), n / (time.time() - began)), flush=True)
    columns[column] = [WORD.sub(lambda m: done[m.group(0)], p) for p in prepped]
    # Self-check: IndicXlit's own sentence function (one model call per word) must give the same text.
    checked = min(PARITY_N, len(sentences))
    own = [engine.translit_sentence(s, "hi") for s in sentences[:checked]]
    mismatches = [i for i in range(checked) if own[i] != columns[column][i]]
    for i in mismatches:
        print("PARITY MISMATCH in %s\n  own  : %s\n  batch: %s" % (column, own[i], columns[column][i]))
    variants[column] = {
        "rescore": rescore, "candidates_per_word": topk,
        "empty_outputs": sum(1 for w in words if not done[w]),
        "parity_checked_sentences": checked, "parity_mismatches": len(mismatches),
    }

meta = {
    "beam_width": BEAM,
    "model_url": indic2en.MODEL_DOWNLOAD_URL, "model_version": indic2en.XLIT_VERSION,
    "distinct_words": len(words),
    "variants": variants,
    "seconds": round(time.time() - start),
    "versions": {
        "python": sys.version.split()[0], "torch": torch.__version__, "fairseq": fairseq.__version__,
        "numpy": numpy.__version__, "ai4bharat-transliteration": version("ai4bharat-transliteration"),
    },
}
with open(dst, "w", encoding="utf-8") as f:
    json.dump({"sentences": sentences, "columns": columns, "meta": meta}, f, ensure_ascii=False)
print("IndicXlit done:", json.dumps(meta))
'''

# IndicXlit imports urduhack (which pulls in TensorFlow) but only calls it for Urdu.
STUB = (
    "import pathlib, site;"
    "pathlib.Path(site.getsitepackages()[0], 'urduhack.py')"
    ".write_text('def normalize(text):\\n    return text\\n')"
)
SETUP = [
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    [sys.executable, "-m", "uv", "venv", "--seed", "--python", "3.8", ENV],
    # pip >= 24.1 rejects omegaconf 2.0.x, which fairseq requires
    [PY, "-m", "pip", "install", "-q", "pip<24.1"],
    [PY, "-m", "pip", "install", "-q", "torch==2.4.1", "torchaudio==2.4.1",
     "--extra-index-url", "https://download.pytorch.org/whl/cpu"],
    [PY, "-m", "pip", "install", "-q", "numpy<1.24", "fairseq==0.12.2", "pydload", "ujson", "tqdm",
     "indic-nlp-library"],
    [PY, "-m", "pip", "install", "-q", "--no-deps", "ai4bharat-transliteration==1.1.3"],
    [PY, "-c", STUB],
]


def run(cmd):
    """Run a command and stream its output into the notebook cell. Returns the exit code."""
    print("$", " ".join(cmd).splitlines()[0][:110], flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


def xlit_env_ready():
    check = [PY, "-c", "import ai4bharat.transliteration, fairseq, urduhack"]
    return os.path.exists(PY) and subprocess.run(check, capture_output=True).returncode == 0


def make_xlit(sentences):
    if not xlit_env_ready():
        shutil.rmtree(ENV, ignore_errors=True)
        if any(run(cmd) for cmd in SETUP):
            return None
    with open("data/xlit_input.json", "w", encoding="utf-8") as f:
        json.dump(sentences, f, ensure_ascii=False)
    with open("data/xlit_worker.py", "w", encoding="utf-8") as f:
        f.write(WORKER)
    if run([PY, "data/xlit_worker.py", "data/xlit_input.json", "data/xlit_output.json", "data/xlit_cache.json"]):
        return None
    with open("data/xlit_output.json", encoding="utf-8") as f:
        return json.load(f)


def saved_xlit(sentences):
    """Fallback when IndicXlit cannot run here: reuse output saved by an earlier run on the same sentences."""
    for path in ["data/xlit_output.json"] + glob.glob("/kaggle/input/*/xlit_output.json"):
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                saved = json.load(f)
            if saved.get("sentences") == sentences:
                saved["meta"]["source"] = f"reused {path}; IndicXlit did not run in this session"
                print(f"\nUsing saved IndicXlit output from {path}")
                return saved
    return None


columns = ["id", "dev", "hum", "xlit", "itr", "dev_raw", "hum_raw", "itr_cased", "xlit_rescored"]
sentences = df["dev"].tolist()
result = make_xlit(sentences) or saved_xlit(sentences)
if result is None:
    print("\n!!! IndicXlit FAILED and no saved output matches these sentences.")
    print("The file is written WITHOUT the xlit columns. Paste the error above.")
    print("Fallback: put the repository's data/xlit_output.json into data/ here (on Kaggle, add it as a dataset) and rerun this cell.")
    report["xlit"] = "FAILED"
    columns = [c for c in columns if not c.startswith("xlit")]
else:
    for column, sentences in result["columns"].items():
        df[column] = [nfc(s).lower() for s in sentences]
    report["xlit"] = result["meta"]

df[columns].to_csv("data/dakshina_hi_4cond.tsv", sep="\t", index=False)
with open("results/stage2_report.json", "w") as f:
    json.dump(report, f, indent=2)
if result is not None:
    info = {}
    if os.path.exists("versions.json"):
        with open("versions.json") as f:
            info = json.load(f)
    info["indicxlit"] = result["meta"]
    with open("versions.json", "w") as f:
        json.dump(info, f, indent=2)

print("\n=== Stage 2 report ===\n" + json.dumps(report, indent=2))
print("\n=== First 3 rows ===")
for _, row in df[columns].head(3).iterrows():
    for col in columns:
        print(f"{col:>13}: {row[col]}")
    print()


## Stage 3: count tokens with every tokenizer

In [ ]:
# Stage 3: count tokens for every sentence, condition and tokenizer. No special tokens are added.
# Writes results/per_sentence_counts.csv and results/tokenizer_inventory.csv.
import base64
import json
import os
from importlib.metadata import version

import pandas as pd
import tiktoken
import transformers
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoTokenizer

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))
os.makedirs("results", exist_ok=True)
transformers.logging.set_verbosity_error()  # silences "sequence longer than model max length"

# name, label, loader, ID, year the vocabulary first shipped (not the year of the model).
# cl100k_base shipped in tiktoken 0.1.1 (December 2022); Qwen3 reuses the Qwen1.5 vocabulary
# file (January 2024); Tekken shipped with Mistral Nemo (July 2024).
TOKENIZERS = [
    ("gpt2", "GPT-2", "hf", "openai-community/gpt2", 2019),
    ("xlmr", "XLM-R", "hf", "FacebookAI/xlm-roberta-base", 2019),
    ("mt5", "mT5", "hf", "google/mt5-base", 2020),
    ("cl100k_base", "cl100k_base", "tiktoken", "cl100k_base", 2022),
    ("llama2", "Llama 2", "hf", "meta-llama/Llama-2-7b-hf", 2023),
    ("qwen3", "Qwen3", "hf", "Qwen/Qwen3-0.6B", 2024),
    ("llama3", "Llama 3", "hf", "meta-llama/Meta-Llama-3-8B", 2024),
    ("o200k_base", "o200k_base", "tiktoken", "o200k_base", 2024),
    ("mistral_tekken", "Mistral Tekken", "tekken", "mistralai/Mistral-Nemo-Instruct-2407", 2024),
    ("sarvam1", "Sarvam-1", "hf", "sarvamai/sarvam-1", 2024),
    ("gemma3", "Gemma 3", "hf", "google/gemma-3-1b-it", 2025),
    ("llama4", "Llama 4", "hf", "meta-llama/Llama-4-Scout-17B-16E-Instruct", 2025),
    ("sarvam_30b", "Sarvam-30B", "hf", "sarvamai/sarvam-30b", 2026),
]
MAIN = ["dev", "hum", "xlit", "itr"]


def generation(year):
    return "G1" if year < 2023 else "G2" if year < 2025 else "G3"


def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")


def load(loader, ident, token):
    """Returns (function mapping a list of texts to token counts, details for the inventory)."""
    if loader == "tiktoken":
        enc = tiktoken.get_encoding(ident)
        # disallowed_special=(): text that looks like a special token is counted as plain text
        return (lambda texts: [len(x) for x in enc.encode_batch(texts, disallowed_special=())],
                {"commit": "tiktoken " + version("tiktoken"), "vocab_size": enc.n_vocab})
    commit = HfApi().model_info(ident, token=token).sha
    if loader == "tekken":
        # Mistral's own vocabulary file, read the way mistral-common reads it. The converted
        # tokenizer.json in the same repo is known to split text with a wrong regex.
        with open(hf_hub_download(ident, "tekken.json", revision=commit, token=token), encoding="utf-8") as f:
            tekken = json.load(f)
        config = tekken["config"]
        inner = config["default_vocab_size"] - config["default_num_special_tokens"]
        ranks = {base64.b64decode(v["token_bytes"]): v["rank"] for v in tekken["vocab"][:inner]}
        enc = tiktoken.Encoding(name="tekken", pat_str=config["pattern"], mergeable_ranks=ranks, special_tokens={})
        return (lambda texts: [len(x) for x in enc.encode_batch(texts, disallowed_special=())],
                {"commit": commit, "vocab_size": config["default_vocab_size"]})
    # trust_remote_code=False: never run code shipped in a model repo, and never stop to ask
    tok = AutoTokenizer.from_pretrained(ident, revision=commit, token=token, trust_remote_code=False)
    return (lambda texts: [len(x) for x in tok(texts, add_special_tokens=False)["input_ids"]],
            {"commit": commit, "vocab_size": len(tok)})


df = pd.read_csv("data/dakshina_hi_4cond.tsv", sep="\t", dtype=str, keep_default_na=False)
conditions = [c for c in df.columns if c != "id"]
counts = {"id": df["id"].astype(int), "dev_words": df["dev"].str.split().str.len()}
token = hf_token()
print(f"{len(df)} sentences | conditions: {conditions} | HF token found: {bool(token)}\n")

inventory, seen = [], {}
for name, label, loader, ident, year in TOKENIZERS:
    row = {"name": name, "label": label, "gen": generation(year), "year": year, "loader": loader,
           "id": ident, "commit": "", "vocab_size": "", "status": "ok", "duplicate_of": "", "error": ""}
    try:
        count, details = load(loader, ident, token)
        new = {f"{name}__{c}": count(df[c].tolist()) for c in conditions}
        row.update(details)
        counts.update(new)
        # Declared before any data was seen: tokenizers with identical per-sentence counts
        # on every main condition count as one tokenizer.
        key = tuple(n for c in MAIN if c in conditions for n in new[f"{name}__{c}"])
        row["duplicate_of"] = seen.get(key, "")
        seen.setdefault(key, name)
        print(f"OK    {label:15} vocab={row['vocab_size']}  commit={str(row['commit'])[:15]}"
              + (f"  DUPLICATE of {row['duplicate_of']}" if row["duplicate_of"] else ""))
    except Exception as e:
        row.update(status="failed", error=f"{type(e).__name__}: {str(e)[:300]}".replace("\n", " "))
        print(f"FAIL  {label:15} {row['error'][:160]}")
    inventory.append(row)

pd.DataFrame(counts).to_csv("results/per_sentence_counts.csv", index=False)
inventory = pd.DataFrame(inventory)
inventory.to_csv("results/tokenizer_inventory.csv", index=False)

info = {}
if os.path.exists("versions.json"):
    with open("versions.json") as f:
        info = json.load(f)
info["tokenizers"] = inventory.to_dict("records")
with open("versions.json", "w") as f:
    json.dump(info, f, indent=2)

print("\n=== Tokenizer inventory (Appendix A) ===")
print(inventory[["label", "gen", "year", "id", "vocab_size", "commit", "status", "duplicate_of"]].to_string(index=False))
print(f"\nSaved results/per_sentence_counts.csv: {len(df)} rows, {len(counts) - 2} count columns")


## Stage 4: statistics and Tables 1 and 2

In [ ]:
# Stage 4: fertility, Discount, Gap, bootstrap CIs, Wilcoxon + Holm, Kendall's tau.
# Reads results/per_sentence_counts.csv and results/tokenizer_inventory.csv; writes tables to results/.
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))

SEED, RESAMPLES, CHUNK = 20261012, 10_000, 500
ROMAN = ["hum", "xlit", "itr"]

counts = pd.read_csv("results/per_sentence_counts.csv")
inv = pd.read_csv("results/tokenizer_inventory.csv", keep_default_na=False)
inv = inv[(inv["status"] == "ok") & (inv["duplicate_of"] == "")].sort_values(["year", "name"]).reset_index(drop=True)
names = inv["name"].tolist()
EXTRA = ["dev_raw", "hum_raw", "itr_cased", "xlit_rescored"]
have = [c for c in ["dev"] + ROMAN + EXTRA if f"{names[0]}__{c}" in counts.columns]
roman = [c for c in ROMAN if c in have]
machine = [c for c in roman if c != "hum"]

# Paired bootstrap over sentences. Multinomial weights are the same thing as drawing sentences
# with replacement, and one set of resamples is shared by every tokenizer and condition.
cols = ["dev_words"] + [f"{t}__{c}" for t in names for c in have]
X = counts[cols].to_numpy(dtype=float)
n = len(X)
rng = np.random.default_rng(SEED)
boot = np.vstack([rng.multinomial(n, np.full(n, 1 / n), size=CHUNK) @ X for _ in range(RESAMPLES // CHUNK)])
total = dict(zip(cols, X.sum(axis=0)))
resampled = dict(zip(cols, boot.T))


def ratio(numerator, denominator, minus=0.0):
    """Point estimate from corpus totals with a 95% percentile interval from the bootstrap."""
    low, high = np.percentile(resampled[numerator] / resampled[denominator] - minus, [2.5, 97.5])
    return total[numerator] / total[denominator] - minus, low, high


def holm(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order])
    out = np.empty(len(p))
    out[order] = np.minimum(adjusted, 1)
    return out


assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])


def wilcoxon(a, b):
    return 1.0 if (a == b).all() else stats.wilcoxon(a, b).pvalue


def to_markdown(table):
    rows = [list(table.columns)] + table.astype(str).values.tolist()
    lines = ["| " + " | ".join(r) + " |" for r in rows]
    return "\n".join(lines[:1] + ["|" + " --- |" * len(table.columns)] + lines[1:]) + "\n"


def save(table, stem, title):
    table.to_csv(f"results/{stem}.csv", index=False)
    with open(f"results/{stem}.md", "w") as f:
        f.write(to_markdown(table))
    print(f"\n=== {title} ===\n{to_markdown(table)}")


meta = inv[["name", "label", "gen", "year"]]

# Table 1: fertility = tokens per Devanagari word.
fertility = meta.copy()
for c in ["dev"] + roman:
    fertility[c] = [ratio(f"{t}__{c}", "dev_words")[0] for t in names]
table1 = fertility[["label", "gen"] + ["dev"] + roman].round(3)
table1.columns = ["Tokenizer", "Gen"] + [c.upper() for c in ["dev"] + roman]
save(table1, "table1_fertility", "Table 1. Fertility (tokens per Devanagari word), lower is cheaper")

# Discount = T(DEV) / T(c). Above 1 means romanization saves tokens.
discount = meta.copy()
for c in roman:
    values = [ratio(f"{t}__dev", f"{t}__{c}") for t in names]
    discount[f"discount_{c}"], discount[f"discount_{c}_low"], discount[f"discount_{c}_high"] = zip(*values)
discount.to_csv("results/discount.csv", index=False)

# Table 2: Gap = T(HUM) / T(machine) - 1, in percent. Positive means human text costs more.
gap = meta.copy()
table2 = pd.DataFrame({"Tokenizer": inv["label"]})
for c in machine:
    values = [ratio(f"{t}__hum", f"{t}__{c}", minus=1.0) for t in names]
    gap[f"gap_{c}"], gap[f"gap_{c}_low"], gap[f"gap_{c}_high"] = (100 * np.array(v) for v in zip(*values))
    gap[f"wilcoxon_holm_{c}"] = holm([wilcoxon(counts[f"{t}__hum"], counts[f"{t}__{c}"]) for t in names])
    table2[f"Gap vs {c.upper()} (%)"] = gap[f"gap_{c}"].map("{:+.2f}".format)
    table2[f"95% CI ({c.upper()})"] = [f"[{lo:+.2f}, {hi:+.2f}]" for lo, hi in zip(gap[f"gap_{c}_low"], gap[f"gap_{c}_high"])]
    # a p-value too small for a float prints as 0, so show the float limit instead
    table2[f"Wilcoxon p, Holm ({c.upper()})"] = [f"{p:.2e}" if p > 0 else "<1e-300" for p in gap[f"wilcoxon_holm_{c}"]]
gap.to_csv("results/gap.csv", index=False)
if machine:
    save(table2, "table2_gap", "Table 2. Human-versus-machine gap with 95% bootstrap confidence intervals")

summary = {
    "sentences": n,
    "devanagari_words": int(total["dev_words"]),
    "tokenizers": len(names),
    "tokenizer_labels": inv["label"].tolist(),
    "earliest_year": int(inv["year"].min()),
    "bootstrap": {"resamples": RESAMPLES, "seed": SEED},
    "discount_hum_median_by_generation": discount.groupby("gen")["discount_hum"].median().round(3).to_dict(),
    "tokenizers_per_generation": inv["gen"].value_counts().sort_index().to_dict(),
    "discount_hum_below_1": discount.loc[discount["discount_hum"] < 1, "label"].tolist(),
    "discount_hum_ci_entirely_below_1": discount.loc[discount["discount_hum_high"] < 1, "label"].tolist(),
}
for c in machine:
    g = gap[f"gap_{c}"]
    summary[f"gap_{c}_percent"] = {
        "mean": round(g.mean(), 2), "min": round(g.min(), 2), "max": round(g.max(), 2),
        "min_tokenizer": inv["label"][g.idxmin()], "max_tokenizer": inv["label"][g.idxmax()],
    }

# RQ3: does the ranking of tokenizers change between human and machine romanization?
if "xlit" in roman:
    ranks = meta.copy()
    ranks["rank_hum"] = fertility["hum"].rank(method="min").astype(int)
    ranks["rank_xlit"] = fertility["xlit"].rank(method="min").astype(int)
    ranks["shift"] = ranks["rank_xlit"] - ranks["rank_hum"]
    ranks.to_csv("results/rankings.csv", index=False)
    tau, p = stats.kendalltau(fertility["hum"], fertility["xlit"])
    # The same resamples give an interval for tau. Token totals rank exactly like fertility.
    hum_boot = np.column_stack([resampled[f"{t}__hum"] for t in names])
    xlit_boot = np.column_stack([resampled[f"{t}__xlit"] for t in names])
    tau_low, tau_high = np.percentile([stats.kendalltau(h, x)[0] for h, x in zip(hum_boot, xlit_boot)], [2.5, 97.5])
    summary["kendall_tau_hum_vs_xlit"] = {"tau": round(float(tau), 3), "p": float(p),
                                          "ci95": [round(float(tau_low), 3), round(float(tau_high), 3)]}
    summary["most_efficient"] = {"hum": inv["label"][fertility["hum"].idxmin()], "xlit": inv["label"][fertility["xlit"].idxmin()]}
    summary["moved_2_or_more_places"] = ranks.loc[ranks["shift"].abs() >= 2, "label"].tolist()
    print("=== Rankings by fertility (1 = cheapest) ===\n" + ranks.drop(columns="name").to_string(index=False))

# Robustness: case and punctuation kept, ITRANS before lowercasing, IndicXlit with its dictionary re-ranking.
robust = meta.copy()
robust["discount_hum"] = discount["discount_hum"]
robust["discount_hum_raw_text"] = [ratio(f"{t}__dev_raw", f"{t}__hum_raw")[0] for t in names]
robust["fertility_itr"] = fertility["itr"]
robust["fertility_itr_cased"] = [ratio(f"{t}__itr_cased", "dev_words")[0] for t in names]
if "xlit_rescored" in have:
    robust["gap_xlit_percent"] = gap["gap_xlit"]
    robust["gap_xlit_rescored_percent"] = [100 * ratio(f"{t}__hum", f"{t}__xlit_rescored", minus=1.0)[0] for t in names]
    summary["gap_xlit_rescored_percent_mean"] = round(robust["gap_xlit_rescored_percent"].mean(), 2)
robust.round(3).to_csv("results/robustness.csv", index=False)
print("\n=== Robustness ===\n" + robust.drop(columns="name").round(3).to_string(index=False))

text = json.dumps(summary, indent=2, default=lambda o: o.item())  # .item() turns numpy numbers into plain ones
with open("results/summary.json", "w") as f:
    f.write(text)
print("\n=== Discount (DEV / condition) with 95% CI ===\n" + discount.drop(columns="name").round(3).to_string(index=False))
print("\n=== summary.json ===\n" + text)


## Stage 5: Figure 1

In [ ]:
# Stage 5: Figure 1. Discount (DEV / HUM) per tokenizer, oldest vocabulary first, with 95% CI whiskers.
# Reads results/discount.csv; writes results/figure1_discount.pdf and .png (300 dpi).
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))

# Colour plus marker shape per generation, so the figure still reads in greyscale print.
# The three colours were checked for colour-blind separation.
GENERATIONS = {
    "G1": ("#2a78d6", "o", "G1: before 2023"),
    "G2": ("#eb6834", "s", "G2: 2023-2024"),
    "G3": ("#1baf7a", "^", "G3: 2025-2026"),
}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"

d = pd.read_csv("results/discount.csv").sort_values(["year", "name"]).reset_index(drop=True)
plt.rcParams.update({"font.size": 8, "axes.edgecolor": MUTED, "pdf.fonttype": 42})
fig, ax = plt.subplots(figsize=(3.4, 0.27 * len(d) + 1.2), layout="constrained")

ax.axvline(1.0, color=MUTED, linewidth=1, zorder=1, label="1.0: same cost")
for gen, (colour, marker, label) in GENERATIONS.items():
    g = d[d["gen"] == gen]
    if g.empty:
        continue
    whiskers = [g["discount_hum"] - g["discount_hum_low"], g["discount_hum_high"] - g["discount_hum"]]
    ax.errorbar(g["discount_hum"], g.index, xerr=whiskers, fmt=marker, color=colour, markersize=6.5,
                markeredgecolor="white", markeredgewidth=0.8, elinewidth=1.4, capsize=2.5, zorder=3, label=label)

ax.set_yticks(range(len(d)))
ax.set_yticklabels([f"{label} ({year})" for label, year in zip(d["label"], d["year"])], color=INK)
ax.set_ylim(len(d) - 0.5, -0.5)  # oldest at the top
# A ratio belongs on a log axis: halving and doubling the cost sit the same distance from 1.
ax.set_xscale("log")
low, high = d["discount_hum_low"].min() / 1.25, d["discount_hum_high"].max() * 1.25
ticks = [t for t in (0.125, 0.25, 0.5, 1, 2, 4, 8, 16) if low <= t <= high]
ax.set_xlim(min(low, 0.8), max(high, 1.25))
ax.set_xticks(ticks)
ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.minorticks_off()
side = ax.get_xaxis_transform()  # x in data units, y as a fraction of the axis height
ax.text(1.0, 1.01, "romanized costs more  ", transform=side, ha="right", va="bottom", color=MUTED, fontsize=7)
ax.text(1.0, 1.01, "  romanized saves tokens", transform=side, ha="left", va="bottom", color=MUTED, fontsize=7)
ax.set_xlabel("Discount (log scale)\nDEV tokens / HUM tokens", color=INK)
ax.tick_params(axis="x", colors=MUTED)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
fig.legend(loc="outside lower center", ncol=2, frameon=False, handletextpad=0.4, columnspacing=1.2)

fig.savefig("results/figure1_discount.pdf")
fig.savefig("results/figure1_discount.png", dpi=300)
print("Saved results/figure1_discount.pdf and results/figure1_discount.png")
widest = (d["discount_hum_high"] - d["discount_hum_low"]).max()
print(f"Widest 95% interval: {widest:.3f}. If the whiskers hide behind the markers, say so in the caption.")


## Stage 6: sentences for hand labelling

In [ ]:
# Stage 6: export the 50 sentences where HUM and XLIT token counts differ most, for hand labelling.
# Writes data/error_analysis_top50.csv and data/error_analysis_random50.csv, each with an empty "cause" column,
# and results/word_pair_differences.csv with the most frequent spelling differences.
import collections
import os
import sys

import pandas as pd

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))

TOP = 50
df = pd.read_csv("data/dakshina_hi_4cond.tsv", sep="\t", dtype=str, keep_default_na=False)
counts = pd.read_csv("results/per_sentence_counts.csv")
inv = pd.read_csv("results/tokenizer_inventory.csv", keep_default_na=False)
names = inv.loc[(inv["status"] == "ok") & (inv["duplicate_of"] == ""), "name"]
if "xlit" not in df.columns:
    sys.exit("No xlit column: the IndicXlit step in Stage 2 failed, so there is nothing to compare.")
assert (df["id"].astype(int) == counts["id"]).all(), "sentence order differs between the two files"


def differing_words(hum, xlit):
    hum, xlit = hum.split(), xlit.split()
    if len(hum) != len(xlit):
        return "(word counts differ)"
    return "; ".join(f"{h} -> {x}" for h, x in zip(hum, xlit) if h != x)


# Token counts are averaged over the distinct tokenizers, so no single tokenizer picks the sample.
hum = counts[[f"{t}__hum" for t in names]].mean(axis=1)
xlit = counts[[f"{t}__xlit" for t in names]].mean(axis=1)
out = df[["id", "dev", "hum", "xlit"]].copy()
out["dev_words"] = counts["dev_words"]
out["hum_tokens_mean"] = hum.round(2)
out["xlit_tokens_mean"] = xlit.round(2)
out["difference"] = (hum - xlit).round(2)
out["difference_per_word"] = ((hum - xlit) / counts["dev_words"]).round(3)
out["differing_words"] = [differing_words(h, x) for h, x in zip(df["hum"], df["xlit"])]
out["cause"] = ""

top = out.loc[out["difference"].abs().sort_values(ascending=False).index[:TOP]]
# utf-8-sig so that Excel shows Devanagari correctly
top.to_csv("data/error_analysis_top50.csv", index=False, encoding="utf-8-sig")
# The largest differences come from the longest sentences, so a random sample is exported too.
out.sample(TOP, random_state=20261012).to_csv("data/error_analysis_random50.csv", index=False, encoding="utf-8-sig")

# Corpus-wide view of the same question: which word spellings differ most often?
pairs, aligned_words = collections.Counter(), 0
for h, x in zip(df["hum"], df["xlit"]):
    h, x = h.split(), x.split()
    if len(h) == len(x):
        aligned_words += len(h)
        pairs.update((a, b) for a, b in zip(h, x) if a != b)
differing = sum(pairs.values())
common = pd.DataFrame([(a, b, n, round(100 * n / differing, 2)) for (a, b), n in pairs.most_common(TOP)],
                      columns=["hum", "xlit", "count", "percent_of_differing_words"])
common.to_csv("results/word_pair_differences.csv", index=False)
print(f"Words spelled differently by HUM and XLIT: {differing} of {aligned_words} aligned words "
      f"({100 * differing / aligned_words:.1f}%), {len(pairs)} distinct pairs")
print(f"The 10 most frequent pairs cover {common['percent_of_differing_words'].head(10).sum():.1f}% of them:")
print(common.head(10).to_string(index=False))

print(f"\nSaved data/error_analysis_top50.csv and data/error_analysis_random50.csv "
      f"({len(top)} sentences each, tokenizers averaged: {len(names)})")
print(f"HUM costs more in {(top['difference'] > 0).sum()} of them, XLIT costs more in {(top['difference'] < 0).sum()}")
print(f"Median length: {top['dev_words'].median():.0f} words in this sample, {out['dev_words'].median():.0f} in the corpus")
print("\nFirst 5 rows:")
print(top[["id", "dev_words", "difference", "differing_words"]].head(5).to_string(index=False, max_colwidth=110))


## Stage 7: values for the [DATA] placeholders

In [ ]:
# Stage 7 helper: list every [DATA] placeholder in the draft next to the value from this run.
# Reads results/summary.json and results/stage2_report.json; writes results/data_placeholders.md.
import json
import os

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))

with open("results/summary.json") as f:
    s = json.load(f)
with open("results/stage2_report.json") as f:
    stage2 = json.load(f)

medians = s["discount_hum_median_by_generation"]
per_gen = s["tokenizers_per_generation"]
xlit, itr, tau = s.get("gap_xlit_percent"), s.get("gap_itr_percent"), s.get("kendall_tau_hum_vs_xlit")
missing = "NOT AVAILABLE (IndicXlit failed in Stage 2)"


def median(gen):
    return f"{medians[gen]}x (n={per_gen[gen]})" if gen in medians else "no tokenizer in this generation"


below = s["discount_hum_below_1"]
rows = [
    ("Abstract, Section 3", "[DATA: N]", s["sentences"]),
    ("Abstract, Introduction", "[DATA: K]", s["tokenizers"]),
    ("Abstract, Introduction", "[DATA: earliest year]", s["earliest_year"]),
    ("Abstract, 5.2", "[DATA: X]% (HUM vs XLIT, mean over tokenizers)", f"{xlit['mean']:+}" if xlit else missing),
    ("Abstract", "[DATA: min]% to [DATA: max]%",
     f"{xlit['min']:+} ({xlit['min_tokenizer']}) to {xlit['max']:+} ({xlit['max_tokenizer']})" if xlit else missing),
    ("5.2", "[DATA: Y]% (HUM vs ITR, mean over tokenizers)", f"{itr['mean']:+}" if itr else missing),
    ("Abstract", "[DATA: A]x (older tokenizers, G1 median)", median("G1")),
    ("Abstract", "[DATA: B]x (recent tokenizers, G3 median)", median("G3")),
    ("5.3", "median discount G1 / G2 / G3", " / ".join(median(g) for g in ("G1", "G2", "G3"))),
    ("Abstract, 5.3", "[DATA: reverses / does not reverse], tokenizer names",
     f"discount below 1.0 for: {', '.join(below)}" if below else "does not reverse for any tokenizer"),
    ("5.3", "discount below 1.0 with the whole 95% CI below 1.0", ", ".join(s["discount_hum_ci_entirely_below_1"]) or "none"),
    ("Abstract, 5.4", "[DATA: tau] (p)",
     f"{tau['tau']} (p = {tau['p']:.3g}; 95% CI {tau['ci95'][0]} to {tau['ci95'][1]})" if tau else missing),
    ("5.4", "most efficient tokenizer under HUM / under XLIT",
     f"{s['most_efficient']['hum']} / {s['most_efficient']['xlit']}" if tau else missing),
    ("5.4", "tokenizers moving two or more places", (", ".join(s["moved_2_or_more_places"]) or "none") if tau else missing),
    ("Section 3", "sentences dropped as shorter than 5 words", stage2["dropped_under_5_words"]),
    ("Section 3", "source sentences before filtering", stage2["source_sentences"]),
    ("Table 1", "all cells", "results/table1_fertility.md"),
    ("Table 2", "all cells", "results/table2_gap.md" if xlit or itr else missing),
    ("Appendix A", "vocab size and commit per tokenizer", "results/tokenizer_inventory.csv"),
    ("Figure 1", "chart", "results/figure1_discount.pdf"),
]
lines = ["| Where in the draft | Placeholder | Value from this run |", "| --- | --- | --- |"]
lines += [f"| {where} | {tag} | {value} |" for where, tag, value in rows]
note = ("\nA plus sign means human romanization costs MORE tokens than the machine version; "
        "a minus sign means it costs FEWER, and the sentence in the draft must be reworded.\n")
text = "\n".join(lines) + "\n" + note
with open("results/data_placeholders.md", "w") as f:
    f.write(text)
print(text)


## Last step: outputs.zip

In [ ]:
# Last step: put everything the paper and the repository need into one file, outputs.zip.
import glob
import os
import zipfile

os.chdir(next(d for d in ("/kaggle/working", "/content", ".") if os.path.isdir(d)))

FILES = ["versions.json", "requirements.txt", "data/dakshina_hi_4cond.tsv", "data/xlit_cache.json",
         "data/error_analysis_top50.csv", "data/error_analysis_random50.csv"] + sorted(glob.glob("results/*"))

with zipfile.ZipFile("outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in FILES:
        if os.path.exists(path):
            z.write(path)
            print(f"{os.path.getsize(path):>12,}  {path}")
        else:
            print(f"     MISSING  {path}")
print(f"\noutputs.zip: {os.path.getsize('outputs.zip') / 1e6:.1f} MB")

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("On Kaggle: download outputs.zip from the Output panel on the right.")
